# EDUAI Audio Lab · Chatterbox Multilingual V3 · Kaggle T4

Este notebook genera audio usando **tu propia voz con consentimiento**. Kaggle no es un servidor permanente: cada sesión requiere encender la GPU manualmente.

1. **Settings → Accelerator → GPU T4 x2**, **Settings → Turn on internet**.
2. Guarda el notebook como **Private**. No publiques muestras vocales ni salidas.
3. Ejecuta las celdas en orden. La instalación se realiza en un **entorno aislado Python 3.11** para proteger las dependencias que Kaggle ya tiene preinstaladas.
4. Sube una grabación de tu voz de **10–30 segundos** y escribe el texto deseado.
5. Descarga **eduai_voz_kaggle.zip** desde la celda final o desde el panel lateral **Output → /kaggle/working**.
6. En EDUAI → Audio Lab → Mis voces → tu perfil, importa el **WAV**, no el ZIP. El perfil de voz y el consentimiento ya persisten en EDUAI.

No se necesita una API key, ni se abre un servidor de Kaggle. El modelo base es Chatterbox V3; la identidad vocal se obtiene mediante una **referencia guardada**, no entrenando pesos nuevos.

In [ ]:
# 1. Comprobar GPU T4 y disponibilidad de Internet
import torch, socket, sys

assert torch.cuda.is_available(), "Activa Settings → Accelerator → GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0))
print("Kaggle Python:", sys.version.split()[0])

for host in ("pypi.org", "github.com", "modelscope.cn"):
    try:
        print(host, "DNS OK:", socket.getaddrinfo(host, 443)[0][4][0])
    except OSError as exc:
        raise RuntimeError(
            f"{host} sin acceso: activa Settings → Turn on internet y reinicia."
        ) from exc

In [ ]:
# 2. Crear Python 3.11 aislado (una vez por sesión)
import sys, subprocess, os
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
ENV_DIR = Path("/kaggle/working/eduai_voice_env")
PY = str(ENV_DIR / "bin" / "python")
if not Path(PY).is_file():
    subprocess.run(UV + ["venv", "--python", "3.11", str(ENV_DIR)], check=True)
print("Python EDUAI:", PY)

In [ ]:
# 3. Instalar pila GPU compatible y Chatterbox V3 SIN modificar el Python base Kaggle
import subprocess, os
from pathlib import Path

env = os.environ.copy()
env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1", UV_HTTP_TIMEOUT="300")

def instalar(*args):
    subprocess.run(
        UV + ["pip", "install", "--python", PY, *args],
        check=True,
        env=env
    )

import importlib.metadata as md_metadata
# Las bibliotecas GPU se instalan en el entorno aislado Python 3.11.
instalar(
    "torch==2.6.0",
    "torchvision==0.21.0",
    "torchaudio==2.6.0",
    "--index-url", "https://download.pytorch.org/whl/cu126"
)
instalar(
    "chatterbox-tts==0.1.7",
    "modelscope>=1.20,<3",
    "setuptools<81"
)
# V3 del código oficial y sin volver a tocar las dependencias CUDA.
instalar(
    "--reinstall", "--no-deps",
    "git+https://github.com/resemble-ai/chatterbox.git@5de7a54aa4e5e2baadb0182dde554908b48b85c2"
)
verificar = r'''
import torch, torchaudio, torchvision, inspect, perth
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
assert torch.cuda.is_available(), "GPU T4 no disponible"
assert torch.__version__.split("+")[0] == "2.6.0"
assert torchaudio.__version__.split("+")[0] == "2.6.0"
assert torchvision.__version__.split("+")[0] == "0.21.0"
assert "t3_model" in inspect.signature(ChatterboxMultilingualTTS.from_local).parameters
assert perth.PerthImplicitWatermarker is not None, "La marca de agua Perth no está disponible"
perth.PerthImplicitWatermarker()
print("CHATTERBOX V3 Y PERTH LISTOS", torch.__version__)
'''
subprocess.run([PY, "-c", verificar], check=True, env=env)

In [ ]:
# 4. Descargar los 4 pesos necesarios desde ModelScope (no desde Hugging Face)
from pathlib import Path
import subprocess

MODEL_DIR = Path("/kaggle/working/eduai_model_v3")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

WEIGHT_FILES = [
    "ve.pt",
    "s3gen.pt",
    "t3_mtl23ls_v3.safetensors",
    "grapheme_mtl_merged_expanded_v1.json"
]
missing = [x for x in WEIGHT_FILES if not (MODEL_DIR / x).is_file()]
if missing:
    subprocess.run([
        str(Path(PY).parent / "modelscope"),
        "download", "--model", "ResembleAI/chatterbox",
        *missing, "--local_dir", str(MODEL_DIR)
    ], check=True, env=env)
for name in WEIGHT_FILES:
    file = MODEL_DIR / name
    assert file.is_file() and file.stat().st_size > 2048, f"Falta: {name}"
print("MODELO V3 DESCARGADO CORRECTAMENTE")

In [ ]:
# 5. Subir tu propia muestra vocal (mantén este notebook privado)
import ipywidgets as widgets
from IPython.display import display

subir_voz = widgets.FileUpload(
    accept=".wav,.mp3,.m4a,.ogg",
    multiple=False,
    description="Subir mi voz"
)
display(subir_voz)

In [ ]:
# 6. Convertir la voz de referencia a WAV mono PCM16 24 kHz
import subprocess, tempfile, wave
from pathlib import Path

entries = subir_voz.value
if not entries:
    raise ValueError("Sube una muestra de tu voz antes de continuar.")
item = next(iter(entries.values())) if isinstance(entries, dict) else entries[0]
content = bytes(item["content"])
if len(content) > 12 * 1024 * 1024:
    raise ValueError("El archivo supera 12 MB.")
extension = Path(item["name"]).suffix or ".wav"
with tempfile.NamedTemporaryFile(suffix=extension, delete=False) as temporary:
    temporary.write(content)
    source_path = Path(temporary.name)
REFERENCIA = Path("/tmp/eduai_referencia.wav")
try:
    subprocess.run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-i", str(source_path), "-ac", "1", "-ar", "24000",
        "-acodec", "pcm_s16le", str(REFERENCIA),
    ], check=True)
finally:
    source_path.unlink(missing_ok=True)
with wave.open(str(REFERENCIA), "rb") as audio:
    duration = audio.getnframes() / audio.getframerate()
if not 10 <= duration <= 30:
    REFERENCIA.unlink(missing_ok=True)
    raise ValueError("La muestra debe durar entre 10 y 30 segundos.")
print("Referencia vocal lista:", round(duration, 1), "segundos")

In [ ]:
# 7. Escribe lo que quieres que diga tu voz
TEXTO = "Hola. Esta es una prueba de mi voz en EDUAI, generada con Chatterbox V3."
assert 1 <= len(TEXTO.strip()) <= 300, "Primera prueba: 1–300 caracteres"

In [ ]:
# 8. Generar WAV en la T4, con marca de agua real Perth
import subprocess, os
from pathlib import Path
from IPython.display import Audio, FileLink, display

SALIDA = Path("/kaggle/working/eduai_voz_kaggle.wav")
script = Path("/tmp/eduai_generar_voz.py")
script.write_text(r'''
import os
import torch
import torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

model = ChatterboxMultilingualTTS.from_local(
    ckpt_dir=os.environ["EDUAI_MODEL_DIR"],
    device="cuda",
    t3_model="v3"
)
with torch.inference_mode():
    waveform = model.generate(
        os.environ["EDUAI_TEXTO"],
        language_id="es",
        audio_prompt_path=os.environ["EDUAI_REFERENCIA"],
    )
torchaudio.save(
    os.environ["EDUAI_SALIDA"],
    waveform.detach().cpu(), model.sr,
    format="wav", encoding="PCM_S", bits_per_sample=16
)
print("VOZ GENERADA CORRECTAMENTE")
''', encoding="utf-8")

execution_env = env.copy()
execution_env.update({
    "EDUAI_MODEL_DIR": str(MODEL_DIR),
    "EDUAI_TEXTO": TEXTO.strip(),
    "EDUAI_REFERENCIA": str(REFERENCIA),
    "EDUAI_SALIDA": str(SALIDA),
})
subprocess.run([PY, str(script)], check=True, env=execution_env)
assert SALIDA.is_file() and SALIDA.stat().st_size > 44
display(Audio(filename=str(SALIDA)))
print("WAV listo para importarlo a EDUAI.")

In [ ]:
# 9. Preparar ZIP para descargar desde Kaggle → Output
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

SALIDA = Path("/kaggle/working/eduai_voz_kaggle.wav")
ZIP_SALIDA = Path("/kaggle/working/eduai_mi_voz.zip")
assert SALIDA.is_file() and SALIDA.stat().st_size > 44, "Primero genera el WAV"
with ZipFile(ZIP_SALIDA, "w", ZIP_DEFLATED) as output:
    output.write(SALIDA, arcname="eduai_voz_kaggle.wav")
print("ZIP preparado:", ZIP_SALIDA)
print("Tamaño:", round(ZIP_SALIDA.stat().st_size / 1024, 1), "KB")
display(FileLink(str(ZIP_SALIDA), result_html_prefix="Descargar ZIP: "))

# Si Kaggle no permite pulsar el enlace, abre Output → /kaggle/working
# y descarga eduai_mi_voz.zip. Guarda una versión con outputs si es necesario.

## Importar en EDUAI y reutilizar la misma voz

1. Descarga **eduai_mi_voz.zip** en el panel **Output** de Kaggle y extrae **eduai_voz_kaggle.wav**.
2. Ve a [EDUAI → Mis voces](https://eduai.innova-space-edu.cl/audio-lab/voices), desbloquea el área administrativa y pulsa **Importar WAV de Kaggle** junto a tu perfil existente («livi»).
3. El audio generado quedará en tu biblioteca privada de EDUAI. La **referencia de tu voz** y el consentimiento permanecen guardados en tu perfil: no tienes que crear otro perfil ni repetir la autorización.
4. Para producir **otra frase** en la misma sesión Kaggle, modifica \`TEXTO\` y ejecuta las celdas 8–9. En una nueva sesión debes volver a encender T4 y cargar la referencia que puedes obtener desde **Descargar referencia** en EDUAI.

**Importante:** tu voz es una referencia personal reutilizable para el modelo general Chatterbox V3; no son pesos entrenados exclusivamente para tu voz. Generar directamente desde la página web sin pasar archivos requiere un trabajador GPU conectado y una cola de trabajos segura, que todavía no están desplegados. No introduzcas contraseñas, JWT o claves administrativas en Kaggle. **Nunca publiques un notebook que contenga grabaciones privadas**.